# Stage 3 - Building the variables

**Input:** `data/processed/sales_clean.csv` (from `02_clean.ipynb`), `data/raw/border_crossings.csv`, and the three INSEE files in `data/raw/insee/`
**Output:** `data/processed/sales_features.csv` (one row per sale, with every variable the models need)

Variables added here:

| Variable | Meaning | Level |
|---|---|---|
| `dist_border_km` | distance to the nearest border crossing (any canton) | sale |
| `dist_geneva_km` | distance to the nearest crossing into the canton of Geneva | sale |
| `dist_annecy_km` | distance to the centre of Annecy, the other price pole of the department | sale |
| `median_income` | median disposable income per consumption unit, 2021 | municipality |
| `log_population` | log of the population in tax households, 2021 | municipality |
| `second_home_share` | share of second homes in the housing stock, 2021: a measure of tourism | municipality |
| `cross_border_share` | share of employed residents who work in Switzerland, 2021 | municipality |
| `log_price_m2` | log of the price per m2: the target of the models | sale |

*Tool declaration: this notebook was drafted with Claude (Anthropic), asked to compute the distance to the border and to merge the INSEE municipality data. The second-home control was added after the raw prices showed that ski resorts, not border towns, are the most expensive municipalities.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from helpers import (DEPARTMENT, PROCESSED, RAW, distance_to_nearest_crossing,
                     haversine_km, load_border_crossings)

ANNECY = (45.8992, 6.1294)  # latitude, longitude of the town centre

sales = pd.read_csv(PROCESSED / "sales_clean.csv", dtype={"code_commune": str})
print(f"{len(sales):,} sales")

## 1. Distance to the border

For each sale I compute the straight-line distance to every border crossing and keep the smallest one.

I build two versions. `dist_border_km` uses all crossings, as in the proposal. `dist_geneva_km` uses only the crossings into the canton of Geneva: the crossings into Valais (Saint-Gingolph, Pas de Morgins, Le Châtelard) are mountain roads leading to far fewer jobs, so being close to them is not the same thing as being close to Geneva. Comparing the two is part of the question.

In [ ]:
crossings = load_border_crossings()
geneva_crossings = crossings[crossings["canton"] == "GE"]

sales["dist_border_km"], sales["nearest_crossing"] = distance_to_nearest_crossing(
    sales["latitude"], sales["longitude"], crossings)
sales["dist_geneva_km"], _ = distance_to_nearest_crossing(
    sales["latitude"], sales["longitude"], geneva_crossings)
sales["dist_annecy_km"] = haversine_km(sales["latitude"], sales["longitude"], *ANNECY)

sales[["dist_border_km", "dist_geneva_km", "dist_annecy_km"]].describe().round(1)

Map check: every sale, coloured by its distance to the border, with the crossings as red crosses. The crossings must sit on the edge of the cloud of points, on the Swiss side.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
points = ax.scatter(sales["longitude"], sales["latitude"], c=sales["dist_border_km"],
                    s=1, cmap="viridis_r")
ax.scatter(crossings["lon"], crossings["lat"], color="red", marker="x", s=60, label="Border crossings")
ax.scatter(ANNECY[1], ANNECY[0], color="black", marker="*", s=120, label="Annecy")
fig.colorbar(points, label="Distance to nearest border crossing (km)")
ax.set_xlabel("Longitude (degrees)")
ax.set_ylabel("Latitude (degrees)")
ax.set_title("Sales in Haute-Savoie and border crossings")
ax.legend()
plt.show()

The coordinates of the crossings were compiled by hand and are accurate to a few hundred metres only. Two checks:

- the sale closest to each crossing must be in the French municipality the crossing is supposed to lead to;
- moving every Geneva crossing by a random error of about 500 m must barely change the distance variable.

In [ ]:
import unicodedata

def plain(text):
    """Lower-case text without accents, to compare municipality names."""
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode().lower()

nearest_sale = sales.loc[sales.groupby("nearest_crossing")["dist_border_km"].idxmin()]
check = crossings.merge(nearest_sale[["nearest_crossing", "commune", "dist_border_km"]],
                        left_on="name", right_on="nearest_crossing")
check["same_municipality"] = [plain(a) == plain(b) for a, b in zip(check["french_side"], check["commune"])]
print(check[["name", "french_side", "commune", "dist_border_km", "same_municipality"]].round(2).to_string(index=False))

rng = np.random.default_rng(0)
changes, correlations = [], []
for _ in range(20):
    moved = geneva_crossings.copy()
    moved["lat"] += rng.normal(0, 0.5 / 110.57, len(moved))   # 0.5 km in degrees of latitude
    moved["lon"] += rng.normal(0, 0.5 / 77.3, len(moved))     # 0.5 km in degrees of longitude at 46 N
    moved_distance, _ = distance_to_nearest_crossing(sales["latitude"], sales["longitude"], moved)
    changes.append(np.abs(moved_distance - sales["dist_geneva_km"]).mean())
    correlations.append(np.corrcoef(moved_distance, sales["dist_geneva_km"])[0, 1])
print(f"\nWith a 500 m error on every crossing: the distance changes by {np.mean(changes):.2f} km on average, "
      f"correlation with the original distance = {np.mean(correlations):.4f}")

## 2. Income and population of the municipality (INSEE Filosofi 2021)

`Q221` is the median disposable income per consumption unit. INSEE hides it (value `s`) for very small municipalities; those are filled with the median of the other municipalities, and flagged.

In [ ]:
filosofi = pd.read_csv(RAW / "insee" / f"filosofi_2021_{DEPARTMENT}.csv", dtype=str)
income = pd.DataFrame({
    "code_commune": filosofi["CODGEO"],
    "median_income": pd.to_numeric(filosofi["Q221"], errors="coerce"),
    "population": pd.to_numeric(filosofi["NBPERS21"], errors="coerce"),
})

sales = sales.merge(income, on="code_commune", how="left", validate="many_to_one")

sales["income_imputed"] = sales["median_income"].isna().astype(int)
print(f"Sales with hidden or missing income: {sales['income_imputed'].sum()} "
      f"in {sales.loc[sales['income_imputed'] == 1, 'commune'].nunique()} municipalities")
sales["median_income"] = sales["median_income"].fillna(income["median_income"].median())
sales["population"] = sales["population"].fillna(income["population"].median())
sales["log_population"] = np.log(sales["population"])

## 3. Share of cross-border workers (INSEE census 2021)

The census file has one row per employed resident, with a weight `IPONDI` (how many people the row represents). The place of work `DCFLT` starts with `SU` when it is in Switzerland. The share is the weighted number of residents working in Switzerland divided by the weighted number of employed residents.

In [ ]:
census = pd.read_csv(RAW / "insee" / f"mobpro_{DEPARTMENT}.csv", dtype=str)
census["weight"] = census["IPONDI"].astype(float)
census["weight_swiss"] = census["weight"] * census["DCFLT"].str.startswith("SU")

by_commune = census.groupby("COMMUNE")[["weight", "weight_swiss"]].sum()
by_commune["cross_border_share"] = by_commune["weight_swiss"] / by_commune["weight"]
print(f"Haute-Savoie: {by_commune['weight_swiss'].sum():,.0f} residents work in Switzerland, "
      f"{by_commune['weight_swiss'].sum() / by_commune['weight'].sum():.1%} of employed residents")

sales = sales.merge(by_commune["cross_border_share"], left_on="code_commune",
                    right_index=True, how="left", validate="many_to_one")
print("Sales without a cross-border share:", sales["cross_border_share"].isna().sum())

## 4. Share of second homes (INSEE census 2021)

The most expensive municipalities of the department are ski resorts and lakeside villages, not border towns. Their prices are driven by tourism and second homes, which has nothing to do with Switzerland. Without a control, a model would read "far from the border and expensive" as a negative border effect. The share of second homes in the housing stock (`P21_RSECOCC / P21_LOG`) measures how touristic a municipality is.

In [ ]:
housing = pd.read_csv(RAW / "insee" / f"logement_2021_{DEPARTMENT}.csv", dtype={"CODGEO": str})
housing["second_home_share"] = housing["P21_RSECOCC"] / housing["P21_LOG"]

sales = sales.merge(housing[["CODGEO", "second_home_share"]], left_on="code_commune",
                    right_on="CODGEO", how="left", validate="many_to_one").drop(columns="CODGEO")
print("Sales without a second-home share:", sales["second_home_share"].isna().sum())

top = sales.groupby("commune")["second_home_share"].first().sort_values(ascending=False)
print("\nMost touristic municipalities:")
print(top.head(8).round(2))

## 5. Target variable and save

In [ ]:
sales["log_price_m2"] = np.log(sales["price_m2"])

assert sales.drop(columns="rooms").notna().all().all(), "unexpected missing values"
sales.to_csv(PROCESSED / "sales_features.csv", index=False)
print(f"Saved {len(sales):,} sales with {sales.shape[1]} columns")
sales.describe().T.round(2)

## 6. First look at the border premium

Median price per m2 by distance band, for the two definitions of distance. This is a raw comparison, with no control for the type of home, the date or the municipality: it only shows what the models will have to explain.

In [ ]:
bands = [0, 2, 5, 10, 15, 20, 30, 60]
first_look = pd.DataFrame({
    "sales (any crossing)": sales.groupby(pd.cut(sales["dist_border_km"], bands), observed=False).size(),
    "median price (any crossing)": sales.groupby(pd.cut(sales["dist_border_km"], bands), observed=False)["price_m2"].median(),
    "sales (Geneva)": sales.groupby(pd.cut(sales["dist_geneva_km"], bands), observed=False).size(),
    "median price (Geneva)": sales.groupby(pd.cut(sales["dist_geneva_km"], bands), observed=False)["price_m2"].median(),
}).round(0)
first_look

In [ ]:
by_commune_sales = sales.groupby("commune").agg(
    sales=("price_m2", "size"),
    median_price_m2=("price_m2", "median"),
    dist_geneva_km=("dist_geneva_km", "median"),
    cross_border_share=("cross_border_share", "first"),
    second_home_share=("second_home_share", "first"),
)
big = by_commune_sales[by_commune_sales["sales"] >= 30]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
points = axes[0].scatter(big["dist_geneva_km"], big["median_price_m2"], s=big["sales"] / 20,
                         c=big["second_home_share"], cmap="coolwarm", alpha=0.8)
fig.colorbar(points, ax=axes[0], label="Share of second homes")
axes[0].set_xlabel("Distance to nearest Geneva crossing (km)")
axes[0].set_ylabel("Median price per m2 (euros)")
axes[0].set_title("Municipalities with at least 30 sales")

axes[1].scatter(big["dist_geneva_km"], big["cross_border_share"], s=big["sales"] / 20, alpha=0.6)
axes[1].set_xlabel("Distance to nearest Geneva crossing (km)")
axes[1].set_ylabel("Share of residents working in Switzerland")
axes[1].set_title("Cross-border workers fade with distance")
plt.tight_layout()
plt.show()

big.sort_values("median_price_m2", ascending=False).head(10).round(2)